In [ ]:
# Ignore all other GPUs except this one
!export CUDA_VISIBLE_DEVICES=0

In [ ]:
import torch
from torch import nn
from torch import optim
from torch.autograd import grad
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import torch.jit as jit

import numpy as np
import matplotlib.pyplot as plt

import os
import time
import pickle
import tqdm
import random
import typing
import itertools
from typing import Union
from datetime import datetime

# import LBFGS
# from LBFGS import FullBatchLBFGS

In [ ]:
def check_size_eq(lst):
    return not any(len(lst[0])!= len(i) for i in lst)
    
def phase_factor(delta):
    return np.exp(1j*delta)

def analytical_solution(x, t, omega, k, phi_amp, delta):

    phi_amp *= phase_factor(delta)

    phi = phi_amp * np.exp(1j * (k*x - omega*t))
    N = -1.0 * k**2 * phi_amp * np.exp(1j * (k*x - omega*t))
    V = -1.0 * k * omega * phi_amp * np.exp(1j * (k*x - omega*t))
    P = (1.0 - omega**2) * phi_amp * np.exp(1j * (k*x - omega*t))

    return phi, N, V, P

def generate_data(xmin, xmax, tmin, tmax, nx, nt, vth, omega_list, phi_amp_list, delta_list):

    if not check_size_eq([omega_list, phi_amp_list, delta_list]):
        raise Exception("Parameter lists are not the same size!")

    x = np.linspace(xmin, xmax, nx)
    t = np.linspace(tmin, tmax, nt)

    x_arr, t_arr = np.meshgrid(x,t)

    phi = np.zeros_like(x_arr, dtype=np.complex128)
    N = np.zeros_like(x_arr, dtype=np.complex128)
    V = np.zeros_like(x_arr, dtype=np.complex128)
    P = np.zeros_like(x_arr, dtype=np.complex128)

    for i in range(len(omega_list)):

        temp_phi, temp_N, temp_V, temp_P = analytical_solution(x_arr, t_arr, omega_list[i], \
                                                               bohm_gross_dispersion(omega_list[i], vth)[0], \
                                                               phi_amp_list[i], delta_list[i])
        phi += temp_phi
        N += temp_N
        V += temp_V
        P += temp_P

    return x_arr.flatten(), t_arr.flatten(), np.real(phi).flatten(), np.real(N).flatten(), np.real(V).flatten(), np.real(P).flatten()

def sparse_measurements(x, t, phi, num_samples):

    indices = np.random.choice(x.shape[0], num_samples, replace=False)

    return x[indices], t[indices], phi[indices]

# def sparse_measurements2(x, t, phi, num_samples):

#     indices = np.random.normal.choice(x.shape[0], num_samples, replace=False)

#     return x[indices], t[indices], phi[indices]


def collocation_points(xmin, xmax, tmin, tmax, Nx, Nt, L, tau):

    x = np.linspace(xmin, xmax, Nx)
    t = np.linspace(tmin, tmax, Nt)

    x_coll, t_coll = np.meshgrid(x, t)

    dx = x[1] - x[0]
    dt = t[1] - t[0]

    return x_coll.flatten(), t_coll.flatten(), dx, dt

Define Our constants (unitless)